In [1]:
!pip install -q schrutepy

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.51.0 requires rich<14,>=12.4.4, but you have rich 15.0.0 which is incompatible.


# **Getting the Data, Formatting It, and Building the Batch Pipeline**

In [2]:
import torch
from schrutepy import schrutepy
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Using device: cuda


In [3]:
df = schrutepy.load_schrute()
df = df.dropna(subset=['character','text'])

lines = []
prev_episode = None

for _, row in df.iterrows():
    if prev_episode is not None and row['episode_name'] != prev_episode:
      lines.append("")
    speaker = str(row['character']).strip()
    dialogue = str(row['text']).strip()
    lines.append(f"{speaker}: {dialogue}")
    prev_episode = row["episode_name"]

text = "\n".join(lines)
print(f"total characters:{len(text):,}")
print(text[:200])

total characters:3,402,917
Michael: All right Jim. Your quarterlies look very good. How are things at the library?
Jim: Oh, I told you. I couldn't close it. So...
Michael: So you've come to the master for guidance? Is this what


In [6]:
!pip install -q tokenizers

from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders

tokenizer = Tokenizer(models.BPE())
tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
tokenizer.decoder = decoders.ByteLevel()

trainer = trainers.BpeTrainer(
    vocab_size=2048,
    special_tokens=["<|endoftext|>"]
)
tokenizer.train_from_iterator([text], trainer=trainer)

vocab_size = tokenizer.get_vocab_size()
EOT_TOKEN = tokenizer.token_to_id("<|endoftext|>")

encode = lambda s: tokenizer.encode(s).ids
decode = lambda l: tokenizer.decode(l)

print(f"Custom Vocab Size: {vocab_size} | EOT ID: {EOT_TOKEN}")
print("Test:", decode(encode("Michael: That's what she said.")))

Custom Vocab Size: 2048 | EOT ID: 0
Test: Michael: That's what she said.


In [7]:
data = torch.tensor(encode(text),dtype=torch.long)
n = int(0.9*len(data))
train_data = data[:n]
val_data = data[n:]

batch_size = 4
block_size = 16

def get_batch(split):
  dataset = train_data if split == "train" else val_data
  ix = torch.randint(len(dataset) - block_size, (batch_size,))
  x = torch.stack([dataset[i : i + block_size] for i in ix])
  y = torch.stack([dataset[i+1 : i + block_size + 1]for i in ix])
  return x.to(device), y.to(device)

xb, yb = get_batch("train")
print("xb shape:",xb.shape)
print("input :", repr(decode(xb[0].tolist())))
print("output :", repr(decode(yb[0].tolist())))

xb shape: torch.Size([4, 16])
input : ' Oh, cool.\nErin: Chocolates.\nDarryl:'
output : ', cool.\nErin: Chocolates.\nDarryl: Why'


# **Embeddings & Causal Self-Attention (with KV Cache)**

In [8]:
import torch.nn as nn
from torch.nn import functional as F

class CausalSelfAttention(nn.Module):
  def __init__(self, n_embed, n_head, dropout=0.1):
    super().__init__()
    assert n_embed % n_head == 0
    self.n_head = n_head
    self.head_dim = n_embed // n_head

    self.c_attn = nn.Linear(n_embed, 3 * n_embed, bias =False)
    self.c_proj = nn.Linear(n_embed, n_embed, bias=False)
    self.dropout = dropout

  def forward(self, x, kv_cache = None):
    B, T, C= x.size()
    qkv = self.c_attn(x)
    q, k, v = qkv.split(C, dim=2)

    q = q.view(B, T, self.n_head, self.head_dim).transpose(1,2)
    k = k.view(B, T, self.n_head, self.head_dim).transpose(1,2)
    v = v.view(B, T, self.n_head, self.head_dim).transpose(1,2)

    if kv_cache is not None:
      past_k, past_v = kv_cache
      k = torch.cat([past_k, k], dim=2)
      v = torch.cat([past_v, v], dim=2)
    new_cache = (k, v)

    is_causal = (T>1) and (kv_cache is None)
    y = F.scaled_dot_product_attention(
        q,k,v,
        attn_mask = None,
        dropout_p = self.dropout if self.training else 0.0,
        is_causal = is_causal
    )

    y = y.transpose(1,2).contiguous().view(B,T,C)
    out = self.c_proj(y)
    return out, new_cache

In [9]:
torch.manual_seed(42)
n_embed = 32
n_head = 4

attn = CausalSelfAttention(n_embed=n_embed, n_head=n_head, dropout=0.0).to(device)
attn.eval()

x_full = torch.randn(1, 5, n_embed, device=device)
out_full, _= attn(x_full, kv_cache=None)

out_prefill, cache = attn(x_full[:,:4,:],kv_cache=None)
out_step5, cache = attn(x_full[:,4:5,:],kv_cache=cache)

print("out_full shape:", out_full.shape)
print("out_prefill shape:", out_prefill.shape)
print("out_step5 shape:", out_step5.shape)

max_diff = (out_full[:,4:5,:] - out_step5).abs().max().item()
print(f"max diff: {max_diff}")

out_full shape: torch.Size([1, 5, 32])
out_prefill shape: torch.Size([1, 4, 32])
out_step5 shape: torch.Size([1, 1, 32])
max diff: 0.0


# **The Transformer Block & Full Language Model**

In [10]:
class FeedForward(nn.Module):
  def __init__(self, n_embed, dropout=0.1):
    super().__init__()
    self.net = nn.Sequential(
        nn.Linear(n_embed, 4 * n_embed),
        nn.GELU(),
        nn.Linear(4 * n_embed, n_embed),
        nn.Dropout(dropout)
    )

  def forward(self, x):
    return self.net(x)

class Block(nn.Module):
  def __init__(self, n_embed, n_head, dropout=0.1):
    super().__init__()
    self.ln1 = nn.LayerNorm(n_embed)
    self.attn = CausalSelfAttention(n_embed, n_head, dropout=dropout)
    self.ln2 = nn.LayerNorm(n_embed)
    self.ffwd = FeedForward(n_embed, dropout=dropout)

  def forward(self, x, kv_cache=None):
    attn_out, new_cache = self.attn(self.ln1(x), kv_cache = kv_cache)
    x = x + attn_out

    x = x + self.ffwd(self.ln2(x))
    return x, new_cache

In [11]:
class MichaelScottGPT(nn.Module):
  def __init__(self, vocab_size, n_embed, n_head, n_layer, block_size, dropout=0.1):
    super().__init__()
    self.block_size = block_size

    self.token_embedding = nn.Embedding(vocab_size, n_embed)
    self.position_embedding = nn.Embedding(block_size, n_embed)
    self.drop = nn.Dropout(dropout)

    self.blocks = nn.ModuleList([
        Block(n_embed, n_head, dropout=dropout) for _ in range(n_layer)
    ])

    self.ln_f = nn.LayerNorm(n_embed)
    self.lm_head = nn.Linear(n_embed, vocab_size, bias=False)

    self.token_embedding.weight = self.lm_head.weight

    self.apply(self._init_weights)

  def _init_weights(self, module):
    if isinstance(module, nn.Linear):
      torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
      if module.bias is not None:
        torch.nn.init.zeros_(module.bias)
    elif isinstance(module, nn.Embedding):
      torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

  def forward(self, idx, targets=None, kv_cache=None):
    B,T = idx.shape

    past_len = kv_cache[0][0].size(2) if kv_cache is not None else 0
    assert past_len + T <= self.block_size, f"Sequence length {past_len + T} exceeds block_size {self.block_size}"

    pos = torch.arange(past_len, past_len + T, device=idx.device)
    x = self.drop(self.token_embedding(idx) + self.position_embedding(pos))

    new_caches = []
    for i, block in enumerate(self.blocks):
      layer_cache = kv_cache[i] if kv_cache is not None else None
      x, updated_cache = block(x, kv_cache=layer_cache)
      new_caches.append(updated_cache)

    x = self.ln_f(x)
    logits = self.lm_head(x)

    loss = None
    if targets is not None:
      B, T, V = logits.shape
      loss = F.cross_entropy(logits.view(B * T, V), targets.view(B * T))

    return logits, loss, new_caches

In [12]:
test_model = MichaelScottGPT(
    vocab_size=vocab_size,
    n_embed=64,
    n_head=4,
    n_layer=4,
    block_size=16,
    dropout=0.1
).to(device)

logits, loss, caches = test_model(xb,yb)

total_params = sum(p.numel() for p in test_model.parameters())
print(f"total parameters: {total_params}")
print(f"Logits shape     : {logits.shape}")
print(f"Initial loss     : {loss.item():.4f} (Expected ~4.57)")
print(f"Number of cached layers: {len(caches)} | Layer 0 K shape: {caches[0][0].shape}")

total parameters: 331136
Logits shape     : torch.Size([4, 16, 2048])
Initial loss     : 7.6522 (Expected ~4.57)
Number of cached layers: 4 | Layer 0 K shape: torch.Size([4, 4, 16, 16])


# **Scaling Up & Training Michael Scott AI**

In [25]:
import math
import time

batch_size = 64
block_size = 256
n_embed = 256
n_head = 8
n_layer = 6
dropout = 0.15

max_iters = 2500
eval_interval = 250
eval_iters = 50
max_lr = 6e-4
min_lr = 6e-5
warmup_iters = 100

def get_batch(split):
    dataset = train_data if split == "train" else val_data
    ix = torch.randint(len(dataset) - block_size, (batch_size,))
    x = torch.stack([dataset[i : i + block_size] for i in ix])
    y = torch.stack([dataset[i + 1 : i + block_size + 1] for i in ix])
    return x.to(device), y.to(device)

torch.manual_seed(42)
model = MichaelScottGPT(
    vocab_size=vocab_size,
    n_embed=n_embed,
    n_head=n_head,
    n_layer=n_layer,
    block_size=block_size,
    dropout=dropout
).to(device)

print(f"Full model parameters: {sum(p.numel() for p in model.parameters()):,}")

use_bf16 = (device == "cuda") and torch.cuda.is_bf16_supported()
amp_dtype = torch.bfloat16 if use_bf16 else torch.float16
scaler = torch.amp.GradScaler("cuda", enabled=(device == "cuda" and not use_bf16))
print(f"Training precision: {amp_dtype} (GradScaler enabled: {scaler.is_enabled()})")

def get_lr(step):
    if step < warmup_iters:
        return max_lr * (step + 1) / warmup_iters
    decay_ratio = (step - warmup_iters) / (max_iters - warmup_iters)
    coeff = 0.5 * (1.0 + math.cos(math.pi * decay_ratio))
    return min_lr + coeff * (max_lr - min_lr)

@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ["train", "val"]:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            with torch.amp.autocast(device_type=device, dtype=amp_dtype, enabled=(device == "cuda")):
                _, loss, _ = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean().item()
    model.train()
    return out

Full model parameters: 5,322,752
Training precision: torch.bfloat16 (GradScaler enabled: False)


In [26]:
optimizer = torch.optim.AdamW(model.parameters(), lr=max_lr, weight_decay=0.1)

start_time = time.time()
for step in range(max_iters):
    lr = get_lr(step)
    for param_group in optimizer.param_groups:
        param_group["lr"] = lr

    if step % eval_interval == 0 or step == max_iters - 1:
        losses = estimate_loss()
        elapsed = time.time() - start_time
        print(f"Step {step:4d}/{max_iters} | Train Loss: {losses['train']:.4f} | Val Loss: {losses['val']:.4f} | LR: {lr:.2e} | Time: {elapsed:.1f}s")

    xb, yb = get_batch("train")
    optimizer.zero_grad(set_to_none=True)

    with torch.amp.autocast(device_type=device, dtype=amp_dtype, enabled=(device == "cuda")):
        _, loss, _ = model(xb, yb)

    scaler.scale(loss).backward()
    scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    scaler.step(optimizer)
    scaler.update()

print("Training complete!")

Step    0/2500 | Train Loss: 7.7112 | Val Loss: 7.7114 | LR: 6.00e-06 | Time: 13.3s
Step  250/2500 | Train Loss: 4.0647 | Val Loss: 4.2354 | LR: 5.95e-04 | Time: 117.3s
Step  500/2500 | Train Loss: 3.7536 | Val Loss: 3.9761 | LR: 5.64e-04 | Time: 220.8s
Step  750/2500 | Train Loss: 3.5282 | Val Loss: 3.7865 | LR: 5.08e-04 | Time: 324.2s
Step 1000/2500 | Train Loss: 3.3253 | Val Loss: 3.6673 | LR: 4.33e-04 | Time: 427.6s
Step 1250/2500 | Train Loss: 3.1883 | Val Loss: 3.6029 | LR: 3.48e-04 | Time: 531.1s
Step 1500/2500 | Train Loss: 3.0742 | Val Loss: 3.5616 | LR: 2.60e-04 | Time: 634.5s
Step 1750/2500 | Train Loss: 2.9769 | Val Loss: 3.5407 | LR: 1.80e-04 | Time: 738.0s
Step 2000/2500 | Train Loss: 2.9082 | Val Loss: 3.5271 | LR: 1.16e-04 | Time: 841.5s
Step 2250/2500 | Train Loss: 2.8639 | Val Loss: 3.5193 | LR: 7.43e-05 | Time: 945.0s
Step 2499/2500 | Train Loss: 2.8242 | Val Loss: 3.5192 | LR: 6.00e-05 | Time: 1048.0s
Training complete!


In [27]:
checkpoint = {
    "model_state_dict": model.state_dict(),
    "tokenizer": "gpt2",
    "config": {
        "vocab_size": vocab_size,
        "n_embed": n_embed,
        "n_head": n_head,
        "n_layer": n_layer,
        "block_size": block_size,
    }
}
torch.save(checkpoint, "michael_scott_bpe.pt")
print("Saved checkpoint to michael_scott_bpe.pt!")

Saved checkpoint to michael_scott_bpe.pt!


# **Fast KV-Cached Autoregressive Generation**

In [28]:
@torch.no_grad()
def generate_with_kv_cache(model, prompt, max_new_tokens=300, temperature=0.8, top_k=40):
  model.eval()

  prompt_ids = encode(prompt)
  idx = torch.tensor(prompt_ids, dtype=torch.long, device=device).unsqueeze(0)

  idx = idx[:, -model.block_size:]

  logits, _, kv_cache = model(idx, kv_cache=None)

  generated_ids = list(prompt_ids)

  for _ in range(max_new_tokens):
    next_logits = logits[:,-1,:] / max(temperature, 1e-5)

    if top_k is not None:
      v, _= torch.topk(next_logits, min(top_k, next_logits.size(-1)))
      next_logits[next_logits < v[:, [-1]]] = -float("Inf")

    probs = F.softmax(next_logits, dim=-1)
    next_token = torch.multinomial(probs, num_samples=1)
    generated_ids.append(next_token.item())

    current_cache_len = kv_cache[0][0].size(2)
    if current_cache_len >= model.block_size:
      kv_cache = [
          (k[:,:,-(model.block_size - 1):,:], v[:,:,-(model.block_size -1):,:])
          for (k,v) in kv_cache
      ]
    logits, _, kv_cache = model(next_token, kv_cache=kv_cache)

  return decode(generated_ids)

In [29]:
prompt = "Jim: Hey Michael, Jan is on line one and she sounds mad.\nMichael:"

t0 = time.time()
output = generate_with_kv_cache(
    model,
    prompt=prompt,
    max_new_tokens=150,
    temperature=0.75,
    top_k=30
)
dt = time.time() - t0

print(f"Generated 150 chars in {dt:.2f}s ({150 / dt:.1f} chars/sec)\n")
print("=" * 50)
print(output)
print("=" * 50)

Generated 150 chars in 0.55s (271.9 chars/sec)

Jim: Hey Michael, Jan is on line one and she sounds mad.
Michael: Oh, well, it's not like you are having a party.
Jan: I don't think that I am a team.
Michael: Okay, well we're going to have some new life.
Jan: I'm not going to tell you what that Michael is.
Michael: What?
Jan: What are you doing?
Michael: I'm going to talk to him.
Jan: I don't want to her.
Jan: Okay.
Michael: I want you to talk to her, I want you to come here.
Michael: I want you to talk to her.
Jan: I want you to me to me, I want you to talk to me.
Michael: Oh, that


Post training

In [30]:
df_michael_era = df[df["season"] <= 7].dropna(subset=["character", "text"]).reset_index(drop=True)

def get_episode_id(row):
    return (row["season"], row["episode"])

sft_pairs = []
for i in range(1, len(df_michael_era)):
    curr_row = df_michael_era.iloc[i]
    prev_row = df_michael_era.iloc[i - 1]

    if (
        get_episode_id(curr_row) == get_episode_id(prev_row)
        and str(curr_row["character"]).strip() == "Michael"
        and str(prev_row["character"]).strip() != "Michael"
    ):
        context_lines = []
        if i >= 2 and get_episode_id(df_michael_era.iloc[i - 2]) == get_episode_id(curr_row):
            s2 = str(df_michael_era.iloc[i - 2]["character"]).strip()
            t2 = str(df_michael_era.iloc[i - 2]["text"]).strip()
            context_lines.append(f"{s2}: {t2}")

        s1 = str(prev_row["character"]).strip()
        t1 = str(prev_row["text"]).strip()
        context_lines.append(f"{s1}: {t1}")

        prompt_str = "\n".join(context_lines) + "\nMichael:"
        reply_str = " " + str(curr_row["text"]).strip() + "<|endoftext|>"
        sft_pairs.append((prompt_str, reply_str))

print(f"Extracted {len(sft_pairs):,} (Context -> Michael) SFT turns!")
print("\n--- Sample SFT Pair ---")
print("PROMPT:\n" + sft_pairs[10][0])
print("TARGET REPLY:\n" + sft_pairs[10][1])

Extracted 9,206 (Context -> Michael) SFT turns!

--- Sample SFT Pair ---
PROMPT:
Michael: What?
Jim: Nothing.
Michael:
TARGET REPLY:
 OK. All right. See you later.<|endoftext|>


In [31]:
import random

IGNORE_INDEX = -100

sft_examples = []
for prompt_str, reply_str in sft_pairs:
    prompt_ids = encode(prompt_str)
    reply_ids = encode(reply_str)

    full_ids = prompt_ids + reply_ids
    labels = [IGNORE_INDEX] * len(prompt_ids) + reply_ids


    if len(full_ids) <= block_size + 1 and len(reply_ids) > 1:
        x_seq = full_ids[:-1]
        y_seq = labels[1:]

        pad_len = block_size - len(x_seq)
        x_seq = x_seq + [EOT_TOKEN] * pad_len
        y_seq = y_seq + [IGNORE_INDEX] * pad_len

        sft_examples.append((torch.tensor(x_seq, dtype=torch.long), torch.tensor(y_seq, dtype=torch.long)))

random.seed(42)
random.shuffle(sft_examples)
split_idx = int(0.9 * len(sft_examples))
sft_train = sft_examples[:split_idx]
sft_val = sft_examples[split_idx:]

def get_sft_batch(split, batch_size=32):
    pool = sft_train if split == "train" else sft_val
    batch = random.sample(pool, batch_size)
    xb = torch.stack([item[0] for item in batch]).to(device)
    yb = torch.stack([item[1] for item in batch]).to(device)
    return xb, yb

xb_sft, yb_sft = get_sft_batch("train", batch_size=4)
print(f"Usable SFT examples: {len(sft_examples):,}")
print("Sample target labels (notice -100 on prompt & padding):")
print(yb_sft[0][:45].tolist())

Usable SFT examples: 9,188
Sample target labels (notice -100 on prompt & padding):
[-100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, 110, 238, 181, 1057, 14, 0, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100]


In [32]:
sft_iters = 600
sft_lr = 1.5e-4
optimizer = torch.optim.AdamW(model.parameters(), lr=sft_lr, weight_decay=0.05)

model.train()
for step in range(sft_iters):
    if step % 100 == 0 or step == sft_iters - 1:
        model.eval()
        with torch.no_grad():
            val_losses = []
            for _ in range(20):
                vx, vy = get_sft_batch("val", batch_size=32)
                with torch.amp.autocast(device_type=device, dtype=amp_dtype, enabled=(device == "cuda")):
                    _, vloss, _ = model(vx, vy)
                val_losses.append(vloss.item())
        print(f"SFT Step {step:3d}/{sft_iters} | Val Reply Loss: {sum(val_losses)/len(val_losses):.4f}")
        model.train()

    xb, yb = get_sft_batch("train", batch_size=32)
    optimizer.zero_grad(set_to_none=True)
    with torch.amp.autocast(device_type=device, dtype=amp_dtype, enabled=(device == "cuda")):
        _, loss, _ = model(xb, yb)

    scaler.scale(loss).backward()
    scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    scaler.step(optimizer)
    scaler.update()

torch.save(model.state_dict(), "michael_scott_sft.pt")
print("SFT complete and saved!")

SFT Step   0/600 | Val Reply Loss: 3.6948
SFT Step 100/600 | Val Reply Loss: 3.1624
SFT Step 200/600 | Val Reply Loss: 3.1666
SFT Step 300/600 | Val Reply Loss: 3.1480
SFT Step 400/600 | Val Reply Loss: 3.1897
SFT Step 500/600 | Val Reply Loss: 3.1888
SFT Step 599/600 | Val Reply Loss: 3.1869
SFT complete and saved!


In [33]:
@torch.no_grad()
def chat_michael_sft(model, conversation_history, temperature=0.7, top_k=40, rep_penalty=1.15, max_reply_tokens=70):
    model.eval()
    prompt_ids = encode(conversation_history)
    idx = torch.tensor([prompt_ids[-(model.block_size - max_reply_tokens):]], dtype=torch.long, device=device)

    logits, _, kv_cache = model(idx, kv_cache=None)
    reply_ids = []

    for _ in range(max_reply_tokens):
        next_logits = logits[:, -1, :].clone()

        for prev_id in set(reply_ids):
            next_logits[0, prev_id] /= rep_penalty

        next_logits = next_logits / max(temperature, 1e-5)

        if len(reply_ids) < 12:
            next_logits[0, EOT_TOKEN] = -float("Inf")

        if top_k is not None:
            v, _ = torch.topk(next_logits, min(top_k, next_logits.size(-1)))
            next_logits[next_logits < v[:, [-1]]] = -float("Inf")

        probs = F.softmax(next_logits, dim=-1)
        next_token = torch.multinomial(probs, num_samples=1)
        token_id = next_token.item()

        if token_id == EOT_TOKEN:
            break
        piece = decode([token_id])
        if "\n" in piece:
            reply_ids.extend(encode(piece.split("\n")[0]))
            break

        reply_ids.append(token_id)
        if kv_cache[0][0].size(2) >= model.block_size:
            break
        logits, _, kv_cache = model(next_token, kv_cache=kv_cache)

    return decode(reply_ids).strip()

for test_prompt in [
    "Jim: Hey Michael, Toby is in the conference room.\nMichael:",
    "Dwight: Michael, there is an emergency in the warehouse!\nMichael:",
    "Pam: Corporate called, they want to know why we spent $3,000 on magic tricks.\nMichael:"
]:
    reply = chat_michael_sft(model, test_prompt)
    print(test_prompt + " " + reply + "\n" + "-" * 50)

Jim: Hey Michael, Toby is in the conference room.
Michael: I know, but it's not a good idea. Aftern.
--------------------------------------------------
Dwight: Michael, there is an emergency in the warehouse!
Michael: The mouth of King! Carganic.
--------------------------------------------------
Pam: Corporate called, they want to know why we spent $3,000 on magic tricks.
Michael: Oh, please. No. The point is a mitt, that is not what they are making me feel. And I am going to say "I'm not going to be there to be my own phone".
--------------------------------------------------


In [41]:
tokenizer.save("michael_scott_tokenizer.json")

In [39]:
checkpoint = {
    "model_state_dict": model.state_dict(),
    "tokenizer_file": "michael_scott_tokenizer.json",
    "config": {
        "vocab_size": vocab_size,
        "n_embed": n_embed,
        "n_head": n_head,
        "n_layer": n_layer,
        "block_size": block_size,
    }
}
torch.save(checkpoint, "michael_scott_bpe.pt")
print("Saved checkpoint and tokenizer!")

Saved checkpoint and tokenizer!
